# Train an HGT-v2 relative-motion forecaster with three causal TCNs

Run **33_prepare_relative_motion_forecasting.ipynb** first. This notebook uses the prepared 91-feature, 20-frame histories to predict the opposing aircraft's North/West/Up position and velocity relative to the observer at **+1, +2, and +5 seconds**.

The shared HGT-v2 graph encoder reads each frame once. Three independent temporal convolutional networks (TCNs) process the resulting history in parallel, one for each forecast horizon. Each TCN uses notebook 34's four width-96 residual blocks with kernel size 3 and dilations 1, 2, 4, and 8 by default. The 61-frame receptive field covers all 20 observed frames. Each TCN has its own six-component regression head, which reads that section's final time step.

Each head predicts a scaled residual from the constant-relative-velocity baseline. The three TCNs and the shared graph encoder train jointly with masked Smooth-L1. Each available sample-horizon pair has equal weight in the data loss. Per-horizon scales come from notebook 33's training partition.

As in notebook 35, the best checkpoint minimizes the mean normalized RMSE across the three horizons on the `test` partition. Validation is evaluated after checkpoint selection.

In [1]:
from pathlib import Path
import os
import sys

root_override = os.environ.get("BVR_RELATIVE_MOTION_ROOT")
candidates = ([Path(root_override).expanduser()] if root_override else
              [Path.cwd(), *Path.cwd().parents])
ROOT = next((path.resolve() for path in candidates
             if (path / "bvr_behavior_prediction/forecasting/relative_motion_gnn.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set BVR_RELATIVE_MOTION_ROOT to the Trajectory_Classification checkout.")
sys.path.insert(0, str(ROOT))

from bvr_behavior_prediction.notebook_runtime import require_pyarrow
require_pyarrow()
import pandas as pd
import torch
from IPython.display import display
from bvr_behavior_prediction.forecasting.relative_motion import (
    HORIZONS, RelativeMotionData, read_json, sha256, write_json,
)
from bvr_behavior_prediction.forecasting.relative_motion_gnn import (
    HORIZON_TCN_ARCHITECTURE, HORIZON_TCN_MODEL_CONTRACT, default_training_config,
    effective_precision, final_evaluation, make_model, train_model,
)

DATA_ROOT = ROOT / "artifacts/relative_motion_forecasting"
EXPERIMENT_ROOT = ROOT / "artifacts/relative_motion_horizon_tcn_forecasting"
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
data_override = os.environ.get("BVR_RELATIVE_MOTION_DATA")
receipt_path = DATA_ROOT / "latest_dataset.json"
if not data_override and not receipt_path.exists():
    raise FileNotFoundError("Run notebook 33 first or set BVR_RELATIVE_MOTION_DATA.")
receipt = None if data_override else read_json(receipt_path)
DATA = Path(data_override or receipt["directory"]).expanduser().resolve()
if receipt and sha256(DATA / "manifest.json") != receipt["manifest_sha256"]:
    raise ValueError("Prepared data changed after notebook 33 completed.")
data = RelativeMotionData(DATA)

DEVICE = os.environ.get("BVR_RELATIVE_MOTION_DEVICE",
                        "cuda" if torch.cuda.is_available() else "cpu")
CONFIG = default_training_config(HORIZON_TCN_ARCHITECTURE)
CONFIG.update({
    "epochs": int(os.environ.get("BVR_RELATIVE_MOTION_EPOCHS", CONFIG["epochs"])),
    "batch_size": int(os.environ.get("BVR_RELATIVE_MOTION_BATCH_SIZE", CONFIG["batch_size"])),
    "evaluation_batch_size": int(os.environ.get("BVR_RELATIVE_MOTION_EVAL_BATCH_SIZE",
                                                    CONFIG["evaluation_batch_size"])),
    "cpu_threads": int(os.environ.get("BVR_RELATIVE_MOTION_CPU_THREADS",
                                         CONFIG["cpu_threads"])),
})
TRAIN_MODEL = True
RESUME = False
REEVALUATE_SAVED = False
EXISTING_RUN = None  # Required for resume or report-only execution.
if (RESUME or not TRAIN_MODEL) and not EXISTING_RUN:
    raise ValueError("Set EXISTING_RUN when resuming or reporting.")
if EXISTING_RUN and TRAIN_MODEL and not RESUME:
    raise ValueError("Use RESUME=True or clear EXISTING_RUN for a fresh fit.")
stamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%S%fZ")
RUN = (Path(EXISTING_RUN).expanduser().resolve() if EXISTING_RUN else
       EXPERIMENT_ROOT / "runs" / f"relative-motion-hgt-horizon-tcn-s{CONFIG['seed']}_{stamp}")
preview = make_model(CONFIG, data)
device = torch.device(DEVICE)
display(pd.Series({
    "architecture": CONFIG["architecture"],
    "device": str(device),
    "effective_precision": effective_precision(device, CONFIG["precision"]),
    "parameters": sum(parameter.numel() for parameter in preview.parameters()),
    "input_features": len(data.manifest["features"]),
    "history_steps": data.manifest["history_steps"],
    "section_horizons_s": HORIZONS.tolist(),
    "tcn_receptive_field": preview.sections[0].temporal.receptive_field,
    "run": str(RUN),
}))
display(pd.Series(CONFIG))
display(pd.DataFrame(data.manifest["counts"]).T)
del preview
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(device))

architecture                          hgt_v2_horizon_tcn_relative_motion
device                                                              cuda
effective_precision                                                 bf16
parameters                                                        933338
input_features                                                        91
history_steps                                                         20
section_horizons_s                                       [1.0, 2.0, 5.0]
tcn_receptive_field                                                   61
run                    C:\Users\theon\Trajectory_Classification\artif...
dtype: object

architecture                 hgt_v2_horizon_tcn_relative_motion
seed                                                         42
width                                                        96
graph_width                                                  64
graph_layers                                                  2
graph_heads                                                   4
dropout                                                    0.25
threat_cpa_horizon_s                                       30.0
threat_time_scale_s                                         5.0
threat_prior_strength                                       1.0
missile_distance_floor_m                                 1000.0
missile_distance_power                                      1.0
epochs                                                      100
batch_size                                                  256
evaluation_batch_size                                       256
learning_rate                           

,groups,perspective_episodes,physical_flights,windows,valid_by_horizon
train,4000,8000,4000,396556,"[396556, 395692, 388556]"
test,3000,6000,3000,1167032,"[1167032, 1161032, 1143032]"
validation,3000,6000,3000,1200344,"[1200344, 1194344, 1176344]"


GPU: NVIDIA GeForce RTX 5080 Laptop GPU


## Fit the graph encoder and three TCNs

Each epoch trains on only notebook 33's retained, non-overlapping training histories. AdamW, gradient clipping, a five-percent warm-up followed by cosine decay, matrix/tensor L1 regularization and BF16-on-supported-CUDA follow notebook 28's training style. HGT attention reductions and physical graph calculations remain FP32. CPU execution falls back to FP32.

The saved run pins the prepared-data manifest, feature order, target contract, complete configuration, model source, reused HGT source, dependency versions and effective precision. `last.pt` is a complete restart checkpoint; `best.pt` is the test-selected model. A resume refuses changed data, code or settings. Start a fresh run when changing the graph, TCN sections, loss, target scales, batch size, or epoch budget.

In [ ]:
def epoch_output(row, metrics):
    test = metrics.loc[metrics.split.eq("test"), [
        "horizon_s", "position_mae_m", "velocity_mae_mps",
        "baseline_position_mae_m", "baseline_velocity_mae_mps",
    ]]
    print(f"Epoch {row['epoch']}: test normalized RMSE={row['test_normalized_rmse']:.5f}, "
          f"train/test Huber={row['train_huber']:.5f}/{row['test_huber']:.5f}, "
          f"LR={row['learning_rate']:.3g}, best={row['best_epoch']}, "
          f"stale={row['stale_epochs']}, {row['seconds']:.1f}s", flush=True)
    display(test.round(3))

if TRAIN_MODEL:
    history = train_model(data, RUN, CONFIG, device=DEVICE, resume=RESUME,
                          callback=epoch_output)
else:
    history = pd.read_csv(RUN / "history.csv")
status = read_json(RUN / "training_status.json")
if status["status"] not in {"completed", "early_stopped"}:
    raise ValueError("Training has not concluded.")
saved_run = read_json(RUN / "run.json")
if saved_run["model_contract"] != HORIZON_TCN_MODEL_CONTRACT:
    raise ValueError("Select a run from notebook 36.")
if saved_run["data_manifest_sha256"] != sha256(DATA / "manifest.json"):
    raise ValueError("Saved run belongs to another prepared dataset.")
if saved_run["feature_names"] != data.manifest["features"]:
    raise ValueError("Saved feature order differs from notebook 33.")
display(history.tail())
print(status)

Epoch 1: test normalized RMSE=0.88875, train/test Huber=0.33210/0.33798, LR=8.4e-05, best=1, stale=0, 774.8s


,horizon_s,position_mae_m,velocity_mae_mps,baseline_position_mae_m,baseline_velocity_mae_mps
3,1.0,14.244,27.980,16.250,31.748
4,2.0,54.685,53.476,63.256,62.070
5,5.0,326.755,125.886,369.919,143.461


Epoch 2: test normalized RMSE=0.62428, train/test Huber=0.18056/0.18281, LR=0.000138, best=2, stale=0, 379.6s


,horizon_s,position_mae_m,velocity_mae_mps,baseline_position_mae_m,baseline_velocity_mae_mps
3,1.0,9.473,18.823,16.250,31.748
4,2.0,37.899,37.980,63.256,62.070
5,5.0,231.209,91.699,369.919,143.461


Epoch 3: test normalized RMSE=0.52436, train/test Huber=0.12994/0.13133, LR=0.000192, best=3, stale=0, 227.8s


,horizon_s,position_mae_m,velocity_mae_mps,baseline_position_mae_m,baseline_velocity_mae_mps
3,1.0,6.973,14.332,16.250,31.748
4,2.0,29.720,31.036,63.256,62.070
5,5.0,207.187,84.251,369.919,143.461


... (12 outputs left out)

Epoch 10: test normalized RMSE=0.38866, train/test Huber=0.07401/0.07466, LR=0.000298, best=10, stale=0, 722.6s


,horizon_s,position_mae_m,velocity_mae_mps,baseline_position_mae_m,baseline_velocity_mae_mps
3,1.0,4.334,9.496,16.250,31.748
4,2.0,20.221,23.040,63.256,62.070
5,5.0,150.907,67.491,369.919,143.461


Epoch 11: test normalized RMSE=0.38703, train/test Huber=0.07337/0.07409, LR=0.000297, best=11, stale=0, 489.5s


,horizon_s,position_mae_m,velocity_mae_mps,baseline_position_mae_m,baseline_velocity_mae_mps
3,1.0,4.279,9.409,16.250,31.748
4,2.0,20.379,23.189,63.256,62.070
5,5.0,150.257,66.977,369.919,143.461


## Final physical-unit evaluation

Run this section after training to evaluate the selected checkpoint on validation. The selected checkpoint is decoded as `constant-relative-velocity baseline + predicted residual`. Position error is the 3D distance between predicted and true relative positions; velocity error is the Euclidean difference between the predicted and true relative velocity vectors. Component RMSE values show errors on each axis. The constant-relative-velocity columns are the no-learning reference on the identical masked examples.

Prediction files contain sample IDs, masks, model predictions, physical targets and baselines for reproducible downstream diagnostics. They can be large for the full notebook-32 corpus. Set `REEVALUATE_SAVED=True` only when deliberately replacing an existing final evaluation.

In [ ]:
metrics = final_evaluation(data, RUN, device=DEVICE,
                           batch_size=CONFIG["evaluation_batch_size"],
                           overwrite=REEVALUATE_SAVED)
position_ratio = metrics.position_mae_m.divide(metrics.baseline_position_mae_m)
velocity_ratio = metrics.velocity_mae_mps.divide(metrics.baseline_velocity_mae_mps)
metrics["position_mae_improvement_pct"] = (
    100 * (1 - position_ratio)).where(metrics.baseline_position_mae_m > 0)
metrics["velocity_mae_improvement_pct"] = (
    100 * (1 - velocity_ratio)).where(metrics.baseline_velocity_mae_mps > 0)
display(metrics[["split", "horizon_s", "samples", "normalized_rmse",
                 "position_mae_m", "baseline_position_mae_m",
                 "position_mae_improvement_pct", "velocity_mae_mps",
                 "baseline_velocity_mae_mps",
                 "velocity_mae_improvement_pct"]].round(3))

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
axes[0].plot(history.epoch, history.train_huber, label="train")
axes[0].plot(history.epoch, history.test_huber, label="test/selection")
axes[0].axvline(read_json(RUN / "selection.json")["best_epoch"],
                color="black", linestyle=":", label="selected")
axes[0].set(title="Masked normalized Smooth-L1", xlabel="epoch", ylabel="loss")
for split, frame in metrics.groupby("split"):
    axes[1].plot(frame.horizon_s, frame.position_mae_m, marker="o", label=f"{split} model")
    axes[1].plot(frame.horizon_s, frame.baseline_position_mae_m, linestyle="--", alpha=.65,
                 label=f"{split} CV")
    axes[2].plot(frame.horizon_s, frame.velocity_mae_mps, marker="o", label=f"{split} model")
    axes[2].plot(frame.horizon_s, frame.baseline_velocity_mae_mps, linestyle="--", alpha=.65,
                 label=f"{split} CV")
axes[1].set(title="Relative-position error", xlabel="horizon (s)", ylabel="3D MAE (m)")
axes[2].set(title="Relative-velocity error", xlabel="horizon (s)", ylabel="vector MAE (m/s)")
for axis in axes:
    axis.grid(alpha=.25)
    axis.legend(fontsize=8)
fig.tight_layout()
figure_path = RUN / "final_relative_motion_metrics.png"
fig.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

write_json(EXPERIMENT_ROOT / "latest_run.json", {
    "directory": str(RUN), "data": str(DATA),
    "data_manifest_sha256": sha256(DATA / "manifest.json"),
    "run_manifest_sha256": sha256(RUN / "run.json"),
    "selection": read_json(RUN / "selection.json"),
})
print("Completed HGT-v2 + horizon-specific TCN relative-motion run:", RUN)